# Olive Counting System - Google Colab Training
**Based on Chen et al. (2017): "Counting Apples and Oranges with Deep Learning"**

## 3-Stage Pipeline:
1. **FCN Blob Detector**: Segments fruit blobs from background
2. **CNN Counter**: Counts fruits in each blob
3. **Linear Regression**: Maps intermediate count to final count

### Instructions:
1. Upload your `patchify` and `patchify_mask` folders to Google Drive
2. Run cells in order
3. Make sure GPU runtime is enabled: Runtime → Change runtime type → GPU

---

## 0. Check GPU & Mount Google Drive

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → GPU")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("\n✓ Google Drive mounted!")

## 1. Set Data Paths

**Edit the paths below** to match where your `patchify` and `patchify_mask` folders are in Google Drive.

In [ ]:
import os

# ============================================================
# EDIT THESE PATHS to match your Google Drive folder structure
# ============================================================
image_dir = '/content/drive/MyDrive/FYP MATERIAL/patchify'
mask_dir = '/content/drive/MyDrive/FYP MATERIAL/patchify_mask'

# Verify paths exist
assert os.path.exists(image_dir), f"❌ Image folder not found: {image_dir}"
assert os.path.exists(mask_dir), f"❌ Mask folder not found: {mask_dir}"

num_images = len([f for f in os.listdir(image_dir) if f.endswith('.png')])
num_masks = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
print(f"✓ Found {num_images} images and {num_masks} masks")

## 2. Define Models (FCN + CNN + Linear Regression)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np


class FCNBlobDetector(nn.Module):
    """
    Fully Convolutional Network for fruit blob segmentation
    Based on FCN-8s architecture with VGG16 backbone
    """
    def __init__(self):
        super(FCNBlobDetector, self).__init__()
        vgg16 = models.vgg16(pretrained=True)
        features = list(vgg16.features.children())

        self.pool3 = nn.Sequential(*features[:17])
        self.pool4 = nn.Sequential(*features[17:24])
        self.pool5 = nn.Sequential(*features[24:])

        self.fc6 = nn.Conv2d(512, 4096, 7, padding=3)
        self.relu6 = nn.ReLU(inplace=True)
        self.drop6 = nn.Dropout2d(p=0.5)
        self.fc7 = nn.Conv2d(4096, 4096, 1)
        self.relu7 = nn.ReLU(inplace=True)
        self.drop7 = nn.Dropout2d(p=0.5)

        self.score_fr = nn.Conv2d(4096, 2, 1)
        self.score_pool3 = nn.Conv2d(256, 2, 1)
        self.score_pool4 = nn.Conv2d(512, 2, 1)

        self.upscore2 = nn.ConvTranspose2d(2, 2, 4, stride=2, bias=False)
        self.upscore_pool4 = nn.ConvTranspose2d(2, 2, 4, stride=2, bias=False)
        self.upscore8 = nn.ConvTranspose2d(2, 2, 16, stride=8, bias=False)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in [self.upscore2, self.upscore_pool4, self.upscore8]:
            c1, c2, h, w = m.weight.data.size()
            weight = self._get_bilinear_kernel(h)
            m.weight.data.copy_(weight)

    def _get_bilinear_kernel(self, size):
        factor = (size + 1) // 2
        center = factor - 1 if size % 2 == 1 else factor - 0.5
        og = np.ogrid[:size, :size]
        kernel = (1 - abs(og[0] - center) / factor) * (1 - abs(og[1] - center) / factor)
        kernel = torch.FloatTensor(kernel).unsqueeze(0).unsqueeze(0)
        return kernel.repeat(2, 2, 1, 1)

    def forward(self, x):
        pool3 = self.pool3(x)
        pool4 = self.pool4(pool3)
        pool5 = self.pool5(pool4)

        fc6 = self.drop6(self.relu6(self.fc6(pool5)))
        fc7 = self.drop7(self.relu7(self.fc7(fc6)))

        score_fr = self.score_fr(fc7)
        upscore2 = self.upscore2(score_fr)

        score_pool4 = self.score_pool4(pool4)
        min_h = min(score_pool4.size()[2], upscore2.size()[2])
        min_w = min(score_pool4.size()[3], upscore2.size()[3])
        if score_pool4.size()[2] > min_h or score_pool4.size()[3] > min_w:
            dh = score_pool4.size()[2] - min_h
            dw = score_pool4.size()[3] - min_w
            score_pool4 = score_pool4[:, :, dh//2:dh//2+min_h, dw//2:dw//2+min_w]
        if upscore2.size()[2] > min_h or upscore2.size()[3] > min_w:
            dh = upscore2.size()[2] - min_h
            dw = upscore2.size()[3] - min_w
            upscore2 = upscore2[:, :, dh//2:dh//2+min_h, dw//2:dw//2+min_w]

        upscore_pool4 = self.upscore_pool4(score_pool4 + upscore2)

        score_pool3 = self.score_pool3(pool3)
        min_h = min(score_pool3.size()[2], upscore_pool4.size()[2])
        min_w = min(score_pool3.size()[3], upscore_pool4.size()[3])
        if score_pool3.size()[2] > min_h or score_pool3.size()[3] > min_w:
            dh = score_pool3.size()[2] - min_h
            dw = score_pool3.size()[3] - min_w
            score_pool3 = score_pool3[:, :, dh//2:dh//2+min_h, dw//2:dw//2+min_w]
        if upscore_pool4.size()[2] > min_h or upscore_pool4.size()[3] > min_w:
            dh = upscore_pool4.size()[2] - min_h
            dw = upscore_pool4.size()[3] - min_w
            upscore_pool4 = upscore_pool4[:, :, dh//2:dh//2+min_h, dw//2:dw//2+min_w]

        upscore8 = self.upscore8(score_pool3 + upscore_pool4)
        if upscore8.size()[2] != x.size()[2] or upscore8.size()[3] != x.size()[3]:
            dh = upscore8.size()[2] - x.size()[2]
            dw = upscore8.size()[3] - x.size()[3]
            upscore8 = upscore8[:, :, dh//2:dh//2+x.size()[2], dw//2:dw//2+x.size()[3]]

        return upscore8


class CNNCountingNetwork(nn.Module):
    """CNN that counts fruits in a segmented blob"""
    def __init__(self, max_count=20):
        super(CNNCountingNetwork, self).__init__()
        self.max_count = max_count
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128, max_count + 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class LinearRegressionCorrection:
    """Linear regression: Final_Count = a * CNN_Count + b"""
    def __init__(self):
        self.a = 1.0
        self.b = 0.0
        self.trained = False

    def fit(self, cnn_counts, true_counts):
        cnn_counts = np.array(cnn_counts).reshape(-1, 1)
        true_counts = np.array(true_counts)
        X = np.column_stack([cnn_counts, np.ones(len(cnn_counts))])
        theta = np.linalg.lstsq(X, true_counts, rcond=None)[0]
        self.a, self.b = theta[0], theta[1]
        self.trained = True

    def predict(self, cnn_count):
        return max(0, round(self.a * cnn_count + self.b))


print("✓ Models defined")

## 3. Define Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image


class OliveSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.image_files = sorted(list(self.image_dir.glob('*.png')))
        print(f"Found {len(self.image_files)} images in {image_dir}")

        self.valid_pairs = []
        for img_path in self.image_files:
            mask_path = self.mask_dir / img_path.name
            if mask_path.exists():
                self.valid_pairs.append((img_path, mask_path))
        print(f"Found {len(self.valid_pairs)} matching image-mask pairs")

        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.valid_pairs[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image).astype(np.float32) / 255.0
        for i in range(3):
            image[:, :, i] = (image[:, :, i] - self.mean[i]) / self.std[i]
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        mask = Image.open(mask_path).convert('L')
        mask = np.array(mask)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()
        return image, mask


def create_data_loaders(image_dir, mask_dir, batch_size=8, train_split=0.8):
    dataset = OliveSegmentationDataset(image_dir, mask_dir)
    total_size = len(dataset)
    train_size = int(total_size * train_split)
    val_size = total_size - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    print(f"\nDataset split: Training: {train_size}, Validation: {val_size}")
    return train_loader, val_loader


print("✓ Dataset classes defined")

## 4. Create Data Loaders

In [ ]:
train_loader, val_loader = create_data_loaders(
    image_dir=image_dir,
    mask_dir=mask_dir,
    batch_size=8,
    train_split=0.8
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 5. Initialize Model & Move to GPU

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = FCNBlobDetector().to(device)
print(f"✓ FCN model loaded on {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Train FCN Segmentation (30 Epochs)

With Colab GPU, this should take **~30-60 minutes** instead of 10-20 hours on CPU.

In [ ]:
import time

# Training settings
EPOCHS = 30
LR = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

best_iou = 0
train_losses = []
val_ious = []

print("="*60)
print(f"TRAINING FCN SEGMENTATION - {EPOCHS} Epochs")
print(f"Device: {device} | Batches/epoch: {len(train_loader)}")
print("="*60 + "\n")

total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # --- Training ---
    model.train()
    train_loss = 0
    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device).long()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Validation ---
    model.eval()
    total_iou = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            intersection = ((preds == 1) & (masks == 1)).float().sum((1, 2))
            union = ((preds == 1) | (masks == 1)).float().sum((1, 2))
            iou = (intersection / (union + 1e-6)).mean()
            total_iou += iou.item()
    val_iou = total_iou / len(val_loader)
    val_ious.append(val_iou)

    epoch_time = time.time() - epoch_start
    eta = epoch_time * (EPOCHS - epoch - 1)

    # Save best model
    marker = ""
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'fcn_segmenter_best.pth')
        marker = " ★ BEST"

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val IoU: {val_iou:.4f} | "
          f"{epoch_time:.0f}s | ETA: {eta/60:.1f}min{marker}")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"✓ Training complete in {total_time/60:.1f} minutes")
print(f"✓ Best Val IoU: {best_iou:.4f}")
print(f"✓ Model saved to: fcn_segmenter_best.pth")

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(train_losses)+1), train_losses, 'b-', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss over Epochs')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(val_ious)+1), val_ious, 'r-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation IoU')
ax2.set_title('Validation IoU over Epochs')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print(f"Best IoU: {max(val_ious):.4f} at epoch {val_ious.index(max(val_ious))+1}")

## 8. Test Model - Visual Results

In [ ]:
# Load best model
model.load_state_dict(torch.load('fcn_segmenter_best.pth', map_location=device))
model.eval()
print("✓ Best model loaded\n")

# Get validation samples
val_images, val_masks = next(iter(val_loader))
val_images_gpu = val_images.to(device)

with torch.no_grad():
    predictions = model(val_images_gpu)
    pred_masks = torch.argmax(predictions, dim=1).cpu().numpy()

val_masks_np = val_masks.numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

num_show = min(4, len(val_images))
fig, axes = plt.subplots(num_show, 3, figsize=(12, 4*num_show))

for i in range(num_show):
    img = val_images[i].numpy().transpose(1, 2, 0)
    img = (img * std + mean).clip(0, 1)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(val_masks_np[i], cmap='gray')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_masks[i], cmap='gray')
    axes[i, 2].set_title('Predicted Mask')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('segmentation_results.png', dpi=150)
plt.show()

## 9. Calculate Validation Metrics

In [ ]:
# Full validation evaluation
model.eval()
total_iou = 0
total_precision = 0
total_recall = 0
num_samples = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        masks_np = masks.numpy()

        for p, m in zip(preds, masks_np):
            tp = ((p == 1) & (m == 1)).sum()
            fp = ((p == 1) & (m == 0)).sum()
            fn = ((p == 0) & (m == 1)).sum()
            union = tp + fp + fn

            if union > 0:
                total_iou += tp / union
            if tp + fp > 0:
                total_precision += tp / (tp + fp)
            if tp + fn > 0:
                total_recall += tp / (tp + fn)
            num_samples += 1

avg_iou = total_iou / num_samples
avg_precision = total_precision / num_samples
avg_recall = total_recall / num_samples
f1 = 2 * avg_precision * avg_recall / (avg_precision + avg_recall + 1e-6)

print("📊 Validation Metrics:")
print(f"   IoU:       {avg_iou:.4f}")
print(f"   Precision: {avg_precision:.4f}")
print(f"   Recall:    {avg_recall:.4f}")
print(f"   F1 Score:  {f1:.4f}")
print(f"   Samples:   {num_samples}")

## 10. Download Trained Model

Save model to Google Drive and/or download to local machine.

In [ ]:
import shutil

# Save to Google Drive
save_dir = '/content/drive/MyDrive/FYP MATERIAL/trained_models'
os.makedirs(save_dir, exist_ok=True)

shutil.copy('fcn_segmenter_best.pth', os.path.join(save_dir, 'fcn_segmenter_best.pth'))
print(f"✓ Model saved to Google Drive: {save_dir}/fcn_segmenter_best.pth")

# Also download to local machine
from google.colab import files
files.download('fcn_segmenter_best.pth')
print("✓ Download started!")